# Pipeline LLM zero-shot real para explicaciones XAI

Notebook reproducible para ejecutar una condición comparativa zero-shot real, sin recuperación CBR.

En esta versión no se recuperan perfiles, vecinos, similitudes, rankings, preferencias inferidas, ejemplos similares ni contraejemplos.

El modelo recibe:

1. La imagen seleccionada y su descripción.
2. La explicación XAI seleccionada.
3. El perfil básico introducido por el usuario.
4. La petición de feedback si la explicación anterior no convence.



## 1. Imports y rutas

In [9]:
from pathlib import Path
import importlib
import html
import sys
import pandas as pd
import ipywidgets as widgets
from IPython.display import Markdown, display


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "evaluacion_online").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "evaluacion_online/llm_cbr_pipeline.py").exists():
    raise FileNotFoundError("Ejecuta el notebook desde la raiz del proyecto o desde llm_zero_shot.")
for path in [PROJECT_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from evaluacion_online import llm_cbr_pipeline
importlib.reload(llm_cbr_pipeline)

from evaluacion_online.llm_cbr_pipeline import (
    DEFAULT_DESCRIPTIONS,
    METHOD_LABEL,
    PipelineResult,
    build_problem_representation,
    generate_with_ollama,
    get_xai_description,
    save_result,
)

DESCRIPTIONS = DEFAULT_DESCRIPTIONS
OUTPUT_DIR = PROJECT_ROOT / "llm_zero_shot/resultados"
IMAGE_MAP_PATH = PROJECT_ROOT / "base_de_casos/cbr_case_base_outputs/image_metadata_template.csv"

print("Proyecto:", PROJECT_ROOT)
print("Descripciones XAI:", DESCRIPTIONS)
print("Mapa de imagenes:", IMAGE_MAP_PATH)
print("Carpeta de salida:", OUTPUT_DIR)


Proyecto: /Users/haojie/PycharmProjects/TFM-Personalized-XAI
Descripciones XAI: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/generacion_descripcion_XAI/resultados_descripciones_xai/descripciones_por_imagen_xai.csv
Mapa de imagenes: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/base_de_casos/cbr_case_base_outputs/image_metadata_template.csv
Carpeta de salida: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/llm_zero_shot/resultados


## 2. Configuración

In [10]:
RUN_OLLAMA = False
OLLAMA_MODEL = "qwen2.5vl:32b"

image_map_df = pd.read_csv(IMAGE_MAP_PATH).rename(
    columns={"model_predicted_class": "class_label"}
)
image_map_df["description_case_id"] = image_map_df["image_id"].astype(int).map(lambda x: f"image{x:02d}")
descriptions_df = pd.read_csv(DESCRIPTIONS)

IMAGE_ID_TO_DESCRIPTION_CASE = dict(zip(image_map_df["image_id"], image_map_df["description_case_id"]))
IMAGE_ID_TO_LABEL = {
    int(row.image_id): f"{int(row.image_id)} - {row.class_label} ({row.image_label})"
    for row in image_map_df.itertuples(index=False)
}
DEMO_IMAGE_ID = 3

print("RUN_OLLAMA:", RUN_OLLAMA)
print("Modelo:", OLLAMA_MODEL)
display(image_map_df)


RUN_OLLAMA: False
Modelo: qwen2.5vl:32b


,image_id,image_label,original_image_path,domain,class_label,model_confidence,initial_description,vqa_support_description,option_A_type,option_B_type,option_C_type,option_D_type,option_E_type,option_F_type,description_case_id
0,1,img_01,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Aguila,NaN,Ave rapaz posada en guante junto a una persona,Original image used as the reference for compa...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image01
1,2,img_02,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Pato,NaN,A close-up profile of a duck showcases its vib...,The base image shows: A close-up profile of a ...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image02
2,3,img_03,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Serpiente,NaN,A vibrant green snake with black stripes curls...,The base image shows: A vibrant green snake wi...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image03
3,4,img_04,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Araña,NaN,A spider is perched on a textured peach-colore...,The base image shows: A spider is perched on a...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image04
4,5,img_05,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Mariposa,NaN,A butterfly with black wings adorned with oran...,The base image shows: A butterfly with black w...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image05
5,6,img_06,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Tigre,NaN,A majestic white tiger with distinct black str...,The base image shows: A majestic white tiger w...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image06
6,7,img_07,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Pajaro,NaN,A black-and-white bird with a slender neck and...,The base image shows: A black-and-white bird w...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image07
7,8,img_08,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Mariposa,NaN,A vibrant blue butterfly rests on a leafy plan...,The base image shows: A vibrant blue butterfly...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image08
8,9,img_09,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Aguila,NaN,"A bird of prey, with dark brown and white plum...","The base image shows: A bird of prey, with dar...",Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image09
9,10,img_10,/Users/haojie/Desktop/TFM/Imagenes/original/00...,Naturaleza / animales,Elefante,NaN,"A young elephant stands amidst tall grass, its...",The base image shows: A young elephant stands ...,Anchor,Grad-CAM,Integrated Gradients,LIME,Saliency,NaN,image10


## 3. Funciones auxiliares zero-shot

In [11]:
METHOD_TO_OPTION = {
    "anchor": "Opción A",
    "gradcam": "Opción B",
    "integrated_gradients": "Opción C",
    "lime": "Opción D",
    "saliency": "Opción E",
    "none": "Opción F (Ninguna)",
}

VISUAL_METHOD_ORDER = ["anchor", "gradcam", "integrated_gradients", "lime", "saliency"]


def method_to_option(method):
    return METHOD_TO_OPTION.get(method, "Método seleccionado")


def select_next_distinct_method(rejected_methods=None, preferred_method=None):
    rejected_methods = set(rejected_methods or [])
    if preferred_method and preferred_method not in rejected_methods:
        return preferred_method, method_to_option(preferred_method)
    for method in VISUAL_METHOD_ORDER:
        if method not in rejected_methods:
            return method, method_to_option(method)
    return None, None



def selected_checkbox_values(checkboxes):
    return [checkbox.description for checkbox in checkboxes if checkbox.value]


def resolve_image_path(row):
    image_path = Path(str(row.get("image_path", "")))
    if image_path.exists():
        return image_path
    if image_path.name:
        project_path = PROJECT_ROOT / "imagenes" / image_path.parent.name.lower() / image_path.name
        if project_path.exists():
            return project_path
    folder = row.get("folder", "")
    file_name = row.get("file_name", "")
    fallback_path = PROJECT_ROOT / "imagenes" / str(folder).lower() / str(file_name)
    if fallback_path.exists():
        return fallback_path
    return image_path


def get_description_row(description_case_id, method):
    case_rows = descriptions_df[descriptions_df["case_id"].astype(str) == str(description_case_id)]
    method_rows = case_rows[case_rows["method"] == method]
    if method_rows.empty:
        return None
    return method_rows.iloc[0]


def image_widget(path, width="180px"):
    image_path = Path(path)
    fmt = image_path.suffix.lower().lstrip(".").replace("jpg", "jpeg")
    return widgets.Image(
        value=image_path.read_bytes(),
        format=fmt,
        layout=widgets.Layout(width=width),
    )


def build_image_widgets(description_case_id, method):
    widgets_out = []
    original_row = get_description_row(description_case_id, "original")
    if original_row is not None:
        original_path = resolve_image_path(original_row)
        if original_path.exists():
            widgets_out.extend([
                widgets.HTML("<b>Imagen principal original</b>"),
                image_widget(original_path, width="180px"),
            ])
    xai_row = get_description_row(description_case_id, method)
    if xai_row is not None:
        xai_path = resolve_image_path(xai_row)
        if xai_path.exists():
            widgets_out.extend([
                widgets.HTML(f"<b>Visualizacion XAI seleccionada: {html.escape(METHOD_LABEL.get(method, method))}</b>"),
                image_widget(xai_path, width="220px"),
            ])
    return widgets_out


def clean_value(value):
    if pd.isna(value):
        return ""
    return str(value)


def build_zero_shot_query():
    image_id = int(image_w.value)
    description_case_id = IMAGE_ID_TO_DESCRIPTION_CASE[image_id]
    map_row = image_map_df[image_map_df["image_id"] == image_id].iloc[0]
    original_row = get_description_row(description_case_id, "original")
    initial_description = ""
    vqa_support_description = ""
    original_image_path = ""
    if original_row is not None:
        initial_description = clean_value(original_row.get("caption_original", ""))
        vqa_support_description = clean_value(original_row.get("descripcion", ""))
        original_image_path = str(resolve_image_path(original_row))
    if not initial_description:
        initial_description = clean_value(map_row.get("notes", ""))
    return {
        "image_id": image_id,
        "image_label": map_row.get("image_label", f"img_{image_id:02d}"),
        "original_image_path": original_image_path,
        "domain": "Naturaleza / animales",
        "model_predicted_class": map_row.get("class_label", "no indicado"),
        "model_confidence": None,
        "initial_description": initial_description,
        "vqa_support_description": vqa_support_description,
        "age_range": age_w.value,
        "education_level": education_w.value,
        "occupation_raw": ", ".join(selected_checkbox_values(occupation_checks)),
        "ai_knowledge_level": ai_level_w.value,
        "domain_knowledge_level": domain_level_w.value,
    }


def build_zero_shot_prompt(query, method, xai_description, requested_change=None):
    change_block = ""
    if requested_change:
        change_block = (
            "\nEl usuario no quedo convencido con la explicacion anterior. "
            f"Ahora pide: {requested_change}\n"
        )
    return f"""Eres un modulo LLM para explicar una clasificacion de imagen con XAI.
Genera una explicacion final en espanol para el usuario.

Contexto interno, no lo muestres como apartados:

* Clase predicha por el modelo: {query.get('model_predicted_class')}
* Metodo XAI seleccionado: {METHOD_LABEL.get(method, method)}
* Descripcion de la imagen: {query.get('initial_description')}
* Descripcion XAI disponible: {xai_description or 'No hay descripcion XAI precalculada.'}
* Nivel educativo: {query.get('education_level')}
* Conocimiento de IA: {query.get('ai_knowledge_level')}/5
* Conocimiento del dominio: {query.get('domain_knowledge_level')}/5
{change_block}
Instrucciones de salida:

* Da SOLO la explicacion final util para el usuario.
* No menciones CBR, perfiles recuperados, casos vecinos, rankings, IDs, similitudes ni prompts.
* No incluyas apartados de descripcion de la imagen original.
* No incluyas apartados de clasificacion del modelo.
* No incluyas ejemplos de instancias similares.
* No incluyas contraejemplos.
* No incluyas limitaciones, dudas ni falta de evidencia.
* Centrate en que zonas o atributos resalta el metodo XAI y por que apoyan la clase predicha.
* Si falta informacion concreta sobre zonas resaltadas, explica el metodo con cautela sin inventar detalles.
* Usa un texto breve, claro y directo, de 2 a 4 parrafos o 3 puntos como maximo.
* Termina con una frase breve preguntando si la explicacion le convence.
"""


def fallback_zero_shot_explanation(method_label, xai_text):
    return (
        f"Esta explicacion zero-shot usa el metodo {method_label}. "
        "No se han usado perfiles recuperados, vecinos, rankings, similitudes, instancias visuales ni contraejemplos.\n\n"
        f"Descripcion XAI usada como apoyo: {xai_text}\n\n"
        "La explicacion se basa solo en la imagen seleccionada, la visualizacion XAI, las descripciones disponibles y el perfil basico indicado. "
        "Te convence esta explicacion?"
    )


## 4. Experimento interactivo zero-shot

In [12]:
available_image_ids = sorted(image_map_df["image_id"].dropna().astype(int).unique())
image_selector_options = [
    (IMAGE_ID_TO_LABEL.get(image_id, f"Imagen {image_id}"), image_id)
    for image_id in available_image_ids
    if image_id in IMAGE_ID_TO_DESCRIPTION_CASE
]
method_selector_options = [
    (METHOD_LABEL[method], method)
    for method in ["anchor", "gradcam", "integrated_gradients", "lime", "saliency"]
]

image_w = widgets.Dropdown(
    description="Imagen",
    options=image_selector_options,
    value=DEMO_IMAGE_ID if DEMO_IMAGE_ID in available_image_ids else image_selector_options[0][1],
    layout=widgets.Layout(width="650px"),
)
method_w = widgets.Dropdown(
    description="Explicacion",
    options=method_selector_options,
    value="gradcam",
    layout=widgets.Layout(width="650px"),
)
age_w = widgets.Dropdown(
    description="Edad",
    options=["18-24", "25-34", "35-44", "45-54", "55-64", "65 o mas"],
    value="25-34",
)
education_w = widgets.Dropdown(
    description="Estudios",
    options=["Bachillerato/FP", "Grado", "Master", "Doctorado", "Prefiero no contestar"],
    value="Master",
)
occupation_options = [
    "Estudiante",
    "Investigador/a (academico)",
    "Docente / profesor/a",
    "Profesional del sector tecnologico / IA / datos",
    "Profesional de otro sector",
    "Desempleado/a / en busqueda",
    "Prefiero no contestar",
]
occupation_checks = [widgets.Checkbox(value=(option == "Investigador/a (academico)"), description=option, indent=False) for option in occupation_options]
occupation_box = widgets.VBox(occupation_checks)
ai_level_w = widgets.IntSlider(description="IA", min=1, max=5, value=4)
domain_level_w = widgets.IntSlider(description="Dominio", min=1, max=5, value=3)
use_ollama_w = widgets.Checkbox(description="Usar Ollama", value=RUN_OLLAMA)
generate_button = widgets.Button(description="Generar explicacion", button_style="primary")
preview_title = widgets.HTML()
preview_box = widgets.HBox(layout=widgets.Layout(flex_flow="row wrap"))
result_box = widgets.VBox()
feedback_box = widgets.VBox()

state = {"result": None, "iteration": 0, "is_generating": False, "rejected_methods": []}


def update_preview(change=None):
    image_id = int(image_w.value)
    description_case_id = IMAGE_ID_TO_DESCRIPTION_CASE[image_id]
    method = method_w.value
    preview_title.value = (
        f"<b>Previsualizacion zero-shot</b><br>"
        f"image_id: <code>{image_id}</code> · imagen: <code>{html.escape(str(description_case_id))}</code><br>"
        f"Metodo XAI: <b>{html.escape(METHOD_LABEL.get(method, method))}</b>"
    )
    preview_box.children = tuple(build_image_widgets(description_case_id, method))


image_w.observe(update_preview, names="value")
method_w.observe(update_preview, names="value")
update_preview()


def html_title(text, level=3):
    return widgets.HTML(f"<h{level}>{html.escape(str(text))}</h{level}>")


def html_text(text):
    return widgets.HTML(
        "<div style='white-space:pre-wrap; max-width:900px; line-height:1.45'>"
        f"{html.escape(str(text))}"
        "</div>"
    )


def generate_zero_shot_explanation(requested_change=None, rejected_methods=None, preferred_method=None):
    query = build_zero_shot_query()
    method, selected_option = select_next_distinct_method(
        rejected_methods=rejected_methods,
        preferred_method=preferred_method,
    )
    if method is None or selected_option is None:
        return None
    description_case_id = IMAGE_ID_TO_DESCRIPTION_CASE[query["image_id"]]
    xai_text = get_xai_description(
        DESCRIPTIONS,
        method=method,
        description_case_id=description_case_id,
    )
    prompt = build_zero_shot_prompt(
        query,
        method,
        xai_text,
        requested_change=requested_change,
    )
    if use_ollama_w.value:
        explanation = generate_with_ollama(prompt, OLLAMA_MODEL)
    else:
        explanation = fallback_zero_shot_explanation(METHOD_LABEL.get(method, method), xai_text)
    state["iteration"] += 1
    result = PipelineResult(
        timestamp=pd.Timestamp.now().isoformat(timespec="seconds"),
        query=query,
        recommended_option=selected_option,
        recommended_method=method,
        neighbor_case_id="zero_shot",
        neighbor_similarity=0.0,
        xai_description=xai_text,
        prompt=prompt,
        explanation=explanation,
        problem=build_problem_representation(query),
        solution={
            "mode": "real_zero_shot",
            "recommended_method": method,
            "uses_profile_retrieval": False,
            "uses_cbr_retrieval": False,
            "uses_visual_neighbors_as_instances": False,
            "uses_counterexamples": False,
        },
        counterexamples=[],
        requested_change=requested_change,
        iteration=state["iteration"],
    )
    state["result"] = result
    return result


def render_result(result, append=False):
    description_case_id = IMAGE_ID_TO_DESCRIPTION_CASE[result.query["image_id"]]
    children = [] if not append else list(result_box.children)
    children.extend([
        html_title(f"Condicion LLM zero-shot - iteracion {result.iteration}", level=3),
        widgets.HTML("<p><b>Nota:</b> no se usan perfiles recuperados, vecinos, similitudes, rankings, preferencias inferidas, instancias visuales ni contraejemplos.</p>"),
        widgets.HBox(build_image_widgets(description_case_id, result.recommended_method), layout=widgets.Layout(flex_flow="row wrap")),
        html_title("Prompt enviado al LLM", level=3),
        widgets.HTML("<details><summary>Ver prompt</summary><pre style='white-space:pre-wrap'>" + html.escape(result.prompt) + "</pre></details>"),
        html_title(f"Explicacion generada - iteracion {result.iteration}", level=3),
        html_text(result.explanation),
    ])
    result_box.children = tuple(children)


def render_feedback_controls(message=None, free_feedback_mode=False):
    convinced_w = widgets.ToggleButtons(options=[("Si", True), ("No", False)], description="Convence?", value=True)
    acceptance_w = widgets.IntSlider(description="Aceptacion", min=1, max=5, value=4)
    satisfaction_w = widgets.IntSlider(description="Satisfaccion", min=1, max=5, value=4)
    confidence_w = widgets.IntSlider(description="Confianza", min=1, max=5, value=4)
    understanding_w = widgets.IntSlider(description="Comprension", min=1, max=5, value=4)
    positive_box = widgets.VBox([
        widgets.HTML("<b>Evalua la explicacion de 1 a 5</b>"),
        acceptance_w,
        satisfaction_w,
        confidence_w,
        understanding_w,
    ])
    needs_w = widgets.Textarea(
        description="Necesito",
        placeholder="Explica como quieres la solucion o que necesitas que aclare.",
        layout=widgets.Layout(width="850px", height="80px"),
    )
    submit_w = widgets.Button(description="Enviar feedback", button_style="success")
    status_w = widgets.HTML("")

    def refresh_visible_fields(change=None):
        if convinced_w.value:
            positive_box.layout.display = ""
            needs_w.layout.display = "none"
        else:
            positive_box.layout.display = "none"
            needs_w.layout.display = "" if free_feedback_mode else "none"

    def save_negative(current, requested_change=None, alternatives_exhausted=False):
        current.convinced = False
        current.rating = None
        current.acceptance = None
        current.satisfaction = None
        current.confidence = None
        current.understanding = None
        current.requested_change = requested_change
        current.alternatives_exhausted = alternatives_exhausted
        save_result(current, OUTPUT_DIR)

    def on_submit(_):
        current = state.get("result")
        if current is None:
            return
        if convinced_w.value:
            current.convinced = True
            current.acceptance = int(acceptance_w.value)
            current.satisfaction = int(satisfaction_w.value)
            current.confidence = int(confidence_w.value)
            current.understanding = int(understanding_w.value)
            current.rating = int(round((current.acceptance + current.satisfaction + current.confidence + current.understanding) / 4))
            current.requested_change = None
            current.alternatives_exhausted = False
            path = save_result(current, OUTPUT_DIR)
            feedback_box.children = (widgets.HTML(f"Feedback guardado en <code>{html.escape(str(path))}</code>. Experimento completado."),)
            return

        rejected_methods = state.setdefault("rejected_methods", [])
        if current.recommended_method not in rejected_methods:
            rejected_methods.append(current.recommended_method)

        if not free_feedback_mode:
            save_negative(current, requested_change=None, alternatives_exhausted=False)
            status_w.value = "Probando el siguiente tipo de explicacion disponible..."
            new_result = generate_zero_shot_explanation(
                requested_change=None,
                rejected_methods=rejected_methods,
                preferred_method=None,
            )
            if new_result is None:
                feedback_box.children = (
                    render_feedback_controls(
                        "Se han agotado todos los tipos de explicacion disponibles. Explica ahora como quieres la solucion:",
                        free_feedback_mode=True,
                    ),
                )
                return
            render_result(new_result, append=True)
            feedback_box.children = (
                render_feedback_controls("La respuesta anterior se guardo como no convincente. Evalua la nueva explicacion:"),
            )
            return

        requested_change = needs_w.value.strip()
        if not requested_change:
            status_w.value = "<span style='color:#b00020'>Indica como quieres la solucion o que necesitas aclarar.</span>"
            return
        save_negative(current, requested_change=requested_change, alternatives_exhausted=True)
        feedback_box.children = (
            widgets.HTML("Feedback final guardado. Se han agotado los tipos de explicacion y el experimento queda cerrado."),
        )
        return

    convinced_w.observe(refresh_visible_fields, names="value")
    submit_w.on_click(on_submit)
    refresh_visible_fields()
    items = []
    if message:
        items.append(widgets.HTML(f"<b>{html.escape(message)}</b>"))
    items.extend([convinced_w, positive_box])
    if free_feedback_mode:
        items.append(needs_w)
    items.extend([submit_w, status_w])
    return widgets.VBox(items)


def on_generate(_):
    if state.get("is_generating"):
        return
    state["is_generating"] = True
    generate_button.disabled = True
    try:
        state["iteration"] = 0
        state["rejected_methods"] = []
        result = generate_zero_shot_explanation(preferred_method=method_w.value)
        if result is None:
            feedback_box.children = (widgets.HTML("No quedan metodos de explicacion disponibles."),)
            return
        render_result(result, append=False)
        feedback_box.children = (render_feedback_controls(),)
    finally:
        generate_button.disabled = False
        state["is_generating"] = False


generate_button._click_handlers.callbacks = []
generate_button.on_click(on_generate)

display(Markdown("### Formulario zero-shot"))
display(widgets.VBox([
    widgets.HBox([image_w, use_ollama_w]),
    method_w,
    preview_title,
    preview_box,
    widgets.HBox([age_w, education_w]),
    widgets.HTML("<b>Ocupacion</b>"),
    occupation_box,
    widgets.HBox([ai_level_w, domain_level_w]),
    generate_button,
]))
display(result_box, feedback_box)


### Formulario zero-shot

VBox()

VBox()